# Paired-end FASTQ: quality report and trimming

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fbcouto/multimerge-genomics-research/blob/main/notebooks/reads/qc_and_trimming.ipynb)

**The problem.** Before aligning or assembling, look at the reads. Does the quality fall towards the 3'
end? Were short fragments read through into the adapter? Are there many `N`, duplicates, an odd GC
content? Then cut the adapters and the low-quality ends, and drop the pairs that become too short,
keeping read 1 and read 2 in step so that the aligner still sees pairs. This is what FastQC and
cutadapt do; MMGEN's trimming gives the same reads as cutadapt (`--no-indels`).

**What you need.** Paired-end FASTQ files (`R1` and `R2`, plain or `.gz`), or a single-end file.

**What you get.** An HTML quality report per file (opens offline in any browser); trimmed FASTQ files;
a before/after table and plots of quality along the read and of read lengths.

In [ ]:
# Setup. Runs on Google Colab or on your own computer: on Colab, the first run installs MMGEN from the GitHub
# release (about a minute). Tables use pandas and plots matplotlib when installed (Colab has both); without
# them the notebook prints plain text and skips the plots.
import platform, subprocess, sys
from pathlib import Path

RELEASE = "https://github.com/fbcouto/multimerge-genomics-research/releases/download/v2.0.0"
try:
    import multimerge_genomics as mmg
except ImportError:
    if platform.system() != "Linux":
        print("MMGEN is not installed in this Python. Download the wheel for your system from\n"
              "  https://github.com/fbcouto/multimerge-genomics-research/releases/tag/v2.0.0\n"
              "install it with:  python -m pip install <the .whl file>\n"
              "and run this cell again.")
        raise
    print("Installing MMGEN (about a minute)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{RELEASE}/multimerge_genomics-2.0.0-cp310-abi3-manylinux_2_28_x86_64.whl"], check=True)
    import multimerge_genomics as mmg

IN_NOTEBOOK = "ipykernel" in sys.modules
try:
    from IPython.display import display, Markdown
except ImportError:                        # plain Python: print instead
    display, Markdown = print, str
try:
    import pandas as pd
    pd.set_option("display.width", 160)
    pd.set_option("display.max_columns", 30)
except ImportError:
    pd = None
try:
    import matplotlib
    if not IN_NOTEBOOK:
        matplotlib.use("Agg")              # plain Python: save the figures, open no window
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

# Chart colors (a palette checked for color-blind readers): two series, a two-sided scale, neutral ink
BLUE, ORANGE, RED, GRAY, INK = "#2a78d6", "#eb6834", "#e34948", "#a3a29c", "#52514e"


def show(rows, n=25):
    """Shows a list of dictionaries as a table (pandas when available, plain text otherwise)."""
    rows = list(rows)
    if pd is not None:
        display(pd.DataFrame(rows[:n]))
    else:
        for row in rows[:n]:
            print("  ".join(f"{k}: {v}" for k, v in row.items()))
    if len(rows) > n:
        print(f"... and {len(rows) - n} more rows")


def tidy(ax):
    """Recessive axes: no top or right frame, a light grid behind the data."""
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(color="#e4e3df", linewidth=0.8)
    ax.set_axisbelow(True)


def save_figure(fig, path):
    """Saves a figure; shows it in a notebook, closes it in plain Python."""
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print("Figure saved to", path)
    if IN_NOTEBOOK:
        plt.show()
    else:
        plt.close(fig)


print(mmg.engine_info())

## Data

By default, from the research demo pack (downloaded once, about 45 MB): the first 30,000 read pairs (2 x 151 bases,
Illumina) of a public sequencing run (ENA ERR2984773) of the ZymoBIOMICS microbial community standard, a
mock community of known bacteria and yeasts. To use your own reads, set `MY_R1` and `MY_R2`
(`MY_R2 = None` for single-end).

In [ ]:
# Your reads (None = the demo data) and the trimming parameters
MY_R1 = None                 # e.g. "sample_R1.fastq.gz"
MY_R2 = None                 # e.g. "sample_R2.fastq.gz"; None with MY_R1 set = single-end
ADAPTER = "auto"             # "auto" = the known adapter found in the first reads, or a sequence
                             # such as "AGATCGGAAGAGC" (Illumina TruSeq) or "CTGTCTCTTATACACATCT" (Nextera)
QUALITY_CUTOFF = 20          # trim 3' ends below this Phred quality (0 = off)
MIN_LENGTH = 50              # drop a pair when a read ends up shorter than this

import urllib.request, zipfile

DATA = Path("data")                          # the demo files are extracted here
OUT = Path("results") / "qc_and_trimming"         # everything this notebook writes
OUT.mkdir(parents=True, exist_ok=True)


def demo_file(pack, member):
    """One file of a demo pack: downloads the pack once into the working folder and extracts only that file."""
    target = DATA / member
    if not target.exists():
        if not Path(pack).exists():
            print(f"Downloading {pack} from the MMGEN release...")
            urllib.request.urlretrieve(f"{RELEASE}/{pack}", pack + ".part")
            Path(pack + ".part").replace(pack)
        with zipfile.ZipFile(pack) as z:
            z.extract(member, DATA)
    return str(target)


USING_DEMO = MY_R1 is None
R1 = MY_R1 or demo_file("mmgen_demo_research_v4.zip", "research/microbio/ERR2984773_zymo_even_D6300_first30k_R1.fastq.gz")
R2 = demo_file("mmgen_demo_research_v4.zip", "research/microbio/ERR2984773_zymo_even_D6300_first30k_R2.fastq.gz") if USING_DEMO else MY_R2
READS = {"R1": R1, "R2": R2} if R2 else {"R1": R1}
for label, p in READS.items():
    print(f"{label}: {p}")

## Step 1. Quality report

One pass over each file measures the read lengths, the quality at each position, the base composition,
GC content, `N` bases, duplicated reads and the known adapters. Read the warnings: each says what it
means and what to do. The HTML page has the charts (in Colab, download it from the Files panel on the
left and open it in a browser).

In [ ]:
before = {}
rows = []
for label, path in READS.items():
    s = mmg.fastq_report(path, str(OUT / f"{label}.report.html"))
    before[label] = s
    adapter = max(s["adapters"], key=lambda a: a["percent"]) if s["adapters"] else None
    rows.append({"file": label, "reads": s["reads"], "length": f"{s['min_length']}-{s['max_length']}",
                 "mean quality": round(s["mean_quality"], 1), "GC %": round(s["gc_percent"], 1),
                 "N %": round(s["n_percent"], 3), "duplicates %": round(s["duplication_percent"], 2),
                 "top adapter": f"{adapter['name']} ({adapter['percent']:.1f}% of reads)" if adapter else "-"})
    for w in s["warnings"]:
        print(f"{label}: {w}")
show(rows)
print("Reports:", ", ".join(str(OUT / f"{label}.report.html") for label in READS))

## Step 2. Trim adapters and low-quality ends

The order is cutadapt's: first the 3' end is cut where the quality falls below `QUALITY_CUTOFF` (the
same running-sum rule as BWA and cutadapt), then the adapter is searched (up to 10% mismatches, at least
3 bases of it at the read end, so a few reads lose 3-4 bases by chance), then the length filter. A pair
is kept or dropped as a whole, so the two outputs keep the same reads in the same order.

In [ ]:
trimmed = {label: str(OUT / f"{label}.trimmed.fastq.gz") for label in READS}
t = mmg.fastq_trim(READS["R1"], trimmed["R1"], input2=READS.get("R2"), output2=trimmed.get("R2"),
                   adapter=ADAPTER, quality_cutoff=QUALITY_CUTOFF, min_length=MIN_LENGTH)
unit = "pairs" if t["paired"] else "reads"
print(f"Adapter used: {t['adapter']}" + (f" (read 2: {t['adapter2']})" if t["paired"] else ""))
print(f"{unit.capitalize()}: {t['reads_in']:,} in, {t['reads_out']:,} kept ({t['reads_out'] / t['reads_in']:.1%}); "
      f"{t['discarded_too_short']:,} dropped as shorter than {MIN_LENGTH}")
print(f"Reads with adapter: {t['reads_with_adapter']:,} (R1)" + (f", {t['reads_with_adapter2']:,} (R2)" if t["paired"] else ""))
print(f"Bases: {t['bases_in']:,} in, {t['bases_out']:,} out; removed for quality {t['bases_removed_quality']:,}, "
      f"as adapter {t['bases_removed_adapter']:,}")

## Step 3. Before and after

The trimmed files are measured again. After trimming, no adapter should be detected, the mean quality
should rise a little, and the read lengths now vary (reads from short fragments lost their adapter).

In [ ]:
after = {label: mmg.fastq_stats(path) for label, path in trimmed.items()}
rows = []
for label in READS:
    for stage, s in (("before", before[label]), ("after", after[label])):
        adapter = max(s["adapters"], key=lambda a: a["percent"]) if s["adapters"] else None
        rows.append({"file": label, "stage": stage, "reads": s["reads"], "bases": s["bases"],
                     "mean length": round(s["mean_length"], 1), "mean quality": round(s["mean_quality"], 2),
                     "reads with adapter %": round(adapter["percent"], 2) if adapter else 0.0})
show(rows)

if plt is not None:
    fig, axes = plt.subplots(1, len(READS) + 1, figsize=(5 * (len(READS) + 1), 3.6))
    for ax, label in zip(axes, READS):
        for stage, s, color in (("before", before[label], BLUE), ("after", after[label], ORANGE)):
            q = s["per_position"]
            x = range(1, len(q["mean_quality"]) + 1)
            ax.plot(x, q["mean_quality"], color=color, linewidth=1.5, label=f"{stage}: mean")
            if stage == "before":
                ax.fill_between(x, q["q10"], q["q90"], color=color, alpha=0.15, linewidth=0, label="before: 10th-90th percentile")
        ax.set(title=f"{label}: quality along the read", xlabel="position in the read", ylabel="Phred quality")
        ax.legend(frameon=False, fontsize=8, loc="lower left")
        tidy(ax)
    ax = axes[-1]
    hist = after["R1"]["length_histogram"]
    ax.bar([l for l, n in hist], [n for l, n in hist], width=1.0, color=BLUE, linewidth=0)
    ax.set(title="R1 after trimming: read lengths", xlabel="length (bases)", ylabel="reads", yscale="log")
    tidy(ax)
    fig.tight_layout()
    save_figure(fig, OUT / "before_after.png")

## Check the result

In [ ]:
import gzip, itertools


def check(condition, text):
    print(("OK     " if condition else "CHECK  ") + text)


def read_names(path, n=10_000):
    with gzip.open(path, "rt") as f:
        return [line.split()[0].removesuffix("/1").removesuffix("/2") for line in itertools.islice(f, 0, 4 * n, 4)]


check(all(s["reads"] == t["reads_out"] for s in after.values()), "every output holds the reads trimming reported")
check(all(s["likely_adapter"] is None for s in after.values()), "no adapter is detected after trimming")
check(all(s["min_length"] >= MIN_LENGTH for s in after.values()), f"no read is shorter than {MIN_LENGTH}")
if "R2" in trimmed:
    check(read_names(trimmed["R1"]) == read_names(trimmed["R2"]), "R1 and R2 are still in step (same read names, same order)")
check(t["reads_out"] / t["reads_in"] > 0.9, f"most {unit} survive ({t['reads_out'] / t['reads_in']:.1%}); "
      "if not, look at the length filter and the quality cutoff")
if USING_DEMO:
    assert before["R1"]["likely_adapter"] is not None and after["R1"]["likely_adapter"] is None
    assert 0.95 < t["reads_out"] / t["reads_in"] < 1
    print("Demo expectations met.")

## Methods paragraph

A starting text for the methods section of a paper or report, filled in with the version and the parameters
this run used. Edit it to your study, and cite the methods MMGEN reproduces (see References).

In [ ]:
import textwrap
VERSION = mmg.engine_info()["version"]
methods = f"""Read quality was assessed with fastq_report. {'Read pairs' if t['paired'] else 'Reads'} were trimmed with
fastq_trim, which follows cutadapt's algorithm without indels: 3' ends were quality-trimmed at Phred {QUALITY_CUTOFF},
the 3' adapter {t['adapter']} ({'detected automatically' if ADAPTER == 'auto' else 'given'}) was removed allowing
a 10% error rate and a minimum overlap of 3 bases, and {'pairs with a read' if t['paired'] else 'reads'} shorter than
{MIN_LENGTH} bases after trimming were discarded ({t['reads_out']:,} of {t['reads_in']:,} kept). Both steps used
MMGEN version {VERSION} (Couto F. MMGEN: MultiMerge Genomics (version {VERSION}). Zenodo. https://doi.org/10.5281/zenodo.20417492)."""
print(textwrap.fill(" ".join(methods.split()), 110))

## Adapt it to your data

* Nextera or other libraries: `ADAPTER = "auto"` recognizes the common ones; or give the sequence.
* Different adapters on the two mates: call `fastq_trim(..., adapter="...", adapter2="...")`.
* Harder trimming for variant calling (`QUALITY_CUTOFF = 25`, `MIN_LENGTH = 70`); gentler for assembly.
* Also cut the 5' end (`quality_cutoff_5=`), drop reads with many N (`max_n=`) or low mean quality
  (`min_mean_quality=`): see `help(mmg.fastq_trim)`.
* Many samples: loop over the file pairs; each call streams and works for files of any size.

## References

* Andrews S. (2010) FastQC: a quality control tool for high throughput sequence data. Babraham Bioinformatics.
* Martin M. (2011) Cutadapt removes adapter sequences from high-throughput sequencing reads. *EMBnet.journal* 17:10.
* Li H., Durbin R. (2009) Fast and accurate short read alignment with Burrows-Wheeler transform. *Bioinformatics*
  25:1754 (the quality-trimming rule).
* ZymoBIOMICS microbial community standard (Zymo Research, D6300); reads: ENA run ERR2984773.